# ☀️ Solar Filament V8 — CPU Native-Tile Fine-Tuning Lab

**Different model, different supervision, not another threshold-only V7.** Fine-tune your existing legitimate YOLO11n-Seg weights on **high-resolution native-image patches with correctly clipped instance masks**. Compare the resulting tile-specialist detector with your full-disk V4 control and, if available, your 0.34 V7 tile run. Select by grouped multi-annotator validation. Generate the final 2048×2048 COCO-RLE CSV only from live image inference.

**Read first:** The 0.55/4-minute U-Net notebook exported a compressed precomputed submission, not predictions from its network; V8 does NOT use that payload. This notebook does not guarantee 0.55 or winning. CPU fine-tuning is substantially slower than inference-only V7 and can span Kaggle sessions. No GPU required; test labels/overlapping public MAGFiLO labels are not loaded. Avoid leaderboard-driven training or dummy-mask hacks.

**Modes:** `PREPARE` → `TRAIN` → `SCREEN` → `CONFIRM` → `SUBMIT` or `ALL` (resource intensive). Save the entire `/kaggle/working/solar_cpu_v8` output as a Kaggle dataset after every completed session and attach it on the next run. `PREVIOUS_V8_ROOT` restores the previous tiles and checkpoint without downloading/retraining. The new detector checkpoint is optional only in PREPARE and TRAIN modes; scoring requires it.

**Public references:** [Organizer evaluation](https://www.kaggle.com/competitions/filament-segmentation-2026/overview/evaluation) · [Self-evaluation notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook). The local PQ implementation is a proxy; compare it against the organizer's implementation before reporting a research result.

In [ ]:
import os, sys, json, hashlib, random, math, shutil, time, gc, warnings, subprocess, importlib.util
from pathlib import Path
from collections import defaultdict
from itertools import product
import numpy as np, pandas as pd, cv2
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
from IPython.display import display

# ======================= EDIT ONLY THIS CELL =======================
MODE = 'PREPARE'    # PREPARE | TRAIN | SCREEN | CONFIRM | SUBMIT | ALL
DATA_ROOT = Path('/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026')
SOURCE_V4_ROOT = Path('/kaggle/input/datasets/lighittha/solarouts/solar_cpu_v4')
SOURCE_V7_ROOT = Path('/kaggle/input/datasets/lighittha/solarouts/solar_cpu_v7')  # optional replay of 0.34 recipe
PREVIOUS_V8_ROOT = None  # attach earlier V8 output; folder containing v8_contract.json
OUTPUT = Path('/kaggle/working/solar_cpu_v8' if Path('/kaggle/working').exists() else '/mnt/data/solar_cpu_v8')
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE='cpu'; SEED=2026; H=W=2048

# The critical change: train the detector on zoomed native image content.
TILE_SIDE=1280                 # 1280 physical pixels per native tile; 5-window full-disk coverage
TILE_WINDOWS='FIVE'            # FIVE = 4 overlapping corners + center
TRAIN_VIEWS_PER_IMAGE=4        # at most 4/5 candidate native tiles; storage/runtime tradeoff
VAL_VIEWS_PER_IMAGE=2
TRAIN_IMAGE_SIZE=832           # YOLO fine-tuning network input: CPU compromise
PREDICT_IMAGE_SIZE=1024       # tiled test inference
TILE_PREPROCESS='clahe_mix'   # must be identical during tile training and inference
MODEL_START='v4'              # v4 = warm-start from your real trained best.pt; OR 'yolo11s-seg.pt' (larger, requires download)
TOTAL_EPOCHS=32
TRAIN_TIME_HOURS=5.0          # caps one CPU training call; .0 removes time cap
TRAIN_BATCH=2
TRAIN_WORKERS=2
TRAIN_IMGSZ=832
TRAIN_PATIENCE=14

# Post-processing candidate search; only train-annotation tune images used for selection
RAW_CONF=.10; RAW_IOU=.82; GLOBAL_IMGSZ=1024; MAX_DET=65
SCREEN_COUNT=28; SCREEN_TOP=6
PROMOTION_MARGIN=.015         # min absolute tune-PQ gain over available legacy control
RUN_OLD_AUDIT=True            # existing audit has already been seen: diagnostic only
INSTALL_IF_MISSING=True
# ================================================================

random.seed(SEED); np.random.seed(SEED); cv2.setNumThreads(0)
for module, spec in [('ultralytics','ultralytics>=8.3,<9'),('pycocotools','pycocotools'),('yaml','PyYAML')]:
    if importlib.util.find_spec(module) is None:
        if not INSTALL_IF_MISSING:raise ImportError('Missing '+spec)
        subprocess.check_call([sys.executable,'-m','pip','-q','install',spec])
import torch, yaml, ultralytics
from ultralytics import YOLO
from pycocotools import mask as coco
if hasattr(torch,'set_num_threads'):torch.set_num_threads(min(4,os.cpu_count() or 2))
print('Mode',MODE,'CPU',torch.__version__,'ultralytics',ultralytics.__version__,'output',OUTPUT)

## 1 — Source integrity and original grouped split
The previous YOLO11n checkpoint used 513 training observations, with 97 tune and 97 audit. Do **not** randomize a new train/tune split for this already-trained checkpoint. Multiple annotators of one physical image remain in the same split.

In [ ]:
def find_dataset():
    if DATA_ROOT and (DATA_ROOT/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').exists():return DATA_ROOT
    choices=[p.parent.parent for p in Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json')]
    if len(choices)!=1:raise FileNotFoundError(f'Cannot uniquely locate official data: {choices[:8]}')
    return choices[0]

def find_v4_root():
    if SOURCE_V4_ROOT and (SOURCE_V4_ROOT/'split_manifest.json').is_file() and (SOURCE_V4_ROOT/'runs/yolo11n-seg/weights/best.pt').is_file():return SOURCE_V4_ROOT
    choices=[p.parent for p in Path('/kaggle/input').rglob('split_manifest.json') if (p.parent/'runs/yolo11n-seg/weights/best.pt').is_file()]
    if len(choices)!=1:raise FileNotFoundError(f'Cannot uniquely locate V4 training checkpoint: {choices[:8]}')
    return choices[0]

def sha256(path):
    d=hashlib.sha256()
    with open(path,'rb') as f:
        for piece in iter(lambda:f.read(1024*1024),b''):d.update(piece)
    return d.hexdigest()

DATA=find_dataset(); V4=find_v4_root()
SPLIT=json.loads((V4/'split_manifest.json').read_text())
assert set(SPLIT)=={'train','tune','audit'}
ANN=json.loads((DATA/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
TRAIN={p.stem:p for p in (DATA/'train'/'train_images').iterdir() if p.suffix.lower() in ('.jpeg','.jpg','.png')}
TEST={p.stem:p for p in (DATA/'test'/'test_images').iterdir() if p.suffix.lower() in ('.jpeg','.jpg','.png')}
assert len(TEST)==180 and len(TRAIN)==707,(len(TEST),len(TRAIN))
BY_ID=defaultdict(list)
for a in ANN['annotations']:BY_ID[str(a['image_id'])].append(a)
OBS=defaultdict(list)
for meta in ANN['images']:
    stem=Path(meta['file_name']).stem
    if stem in TRAIN:OBS[stem].append({'image_id':str(meta['id']),'anns':BY_ID[str(meta['id'])]})
assert set(OBS)==set(TRAIN)
assert set().union(*(set(SPLIT[k]) for k in SPLIT))==set(OBS)
for a in SPLIT:
    for b in SPLIT:
        if a!=b:assert not ({x[:8] for x in SPLIT[a]} & {x[:8] for x in SPLIT[b]})
V4_WEIGHTS=V4/'runs/yolo11n-seg/weights/best.pt'
V4_SHA=sha256(V4_WEIGHTS); SPLIT_SHA=sha256(V4/'split_manifest.json')
print('Observations',len(TRAIN),'train/tune/audit',{x:len(y) for x,y in SPLIT.items()},'versions',sum(map(len,OBS.values())))
print('V4 SHA256',V4_SHA)

## 2 — Native tile geometry, masked ground truth and representative labels
Keep native JPEG pixels and convert the organizer's permitted polygons to true **per-instance YOLO segmentation contours** after cropping. This is not the same as cropping masks at inference. We choose the most representative of the available annotations for each training observation, using agreement between the supplied train annotation versions only.

In [ ]:
def windows(side=TILE_SIDE,kind=TILE_WINDOWS):
    assert side<=H and kind in ('FOUR','FIVE')
    starts=[(0,0),(W-side,0),(0,H-side),(W-side,H-side)]
    if kind=='FIVE':starts.append(((W-side)//2,(H-side)//2))
    return list(dict.fromkeys(starts))
WINDOWS=windows(); assert len(WINDOWS) in (4,5)

def as_rle(counts):
    if isinstance(counts,dict):
        return {'size':[H,W],'counts':counts['counts'].encode('ascii') if isinstance(counts['counts'],str) else counts['counts']}
    return {'size':[H,W],'counts':counts.encode('ascii') if isinstance(counts,str) else counts}
def asc(rle):return rle['counts'].decode('ascii') if isinstance(rle['counts'],bytes) else str(rle['counts'])
def encoded(mask):return coco.encode(np.asfortranarray(np.asarray(mask,dtype=np.uint8)))

def paths(ann):
    seg=ann.get('segmentation',[])
    if isinstance(seg,dict) or not seg:return []
    if isinstance(seg[0],(int,float)):seg=[seg]
    out=[]
    for polygon in seg:
        try:
            v=np.asarray(polygon,dtype=np.float32).reshape(-1,2)
            if len(v)>=3 and np.isfinite(v).all():out.append(v)
        except (TypeError,ValueError):continue
    return out

def full_mask(ann):
    seg=ann.get('segmentation',[])
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            rr=coco.frPyObjects(seg,H,W)
            return coco.decode(rr).astype(np.uint8)
        return coco.decode(as_rle(seg)).astype(np.uint8)
    mask=np.zeros((H,W),np.uint8)
    for p in paths(ann):cv2.fillPoly(mask,[np.rint(p).astype(np.int32)],1)
    return mask

def coarse_union(record,sz=384):
    out=np.zeros((sz,sz),np.uint8)
    for a in record['anns']:
        for p in paths(a):cv2.fillPoly(out,[np.rint(p*(sz/W)).astype(np.int32)],1)
    return out

def representative(stem):
    versions=OBS[stem]
    if len(versions)==1:return versions[0]
    unions=[coarse_union(rec)>0 for rec in versions]
    scores=[]
    for i,a in enumerate(unions):
        pair=[]
        for j,b in enumerate(unions):
            if i==j:continue
            den=int(a.sum()+b.sum());pair.append((2*int((a&b).sum())/den) if den else 1.0)
        scores.append(float(np.mean(pair)))
    return versions[int(np.argmax(scores))]

# Precompute representative labels only for train and tune. Audit is never used to build targets.
REPS={s:representative(s) for s in sorted(set(SPLIT['train']) | set(SPLIT['tune']))}

def tile_preprocess(gray):
    if TILE_PREPROCESS=='none':return gray
    if TILE_PREPROCESS=='clahe_mix':
        clahe=cv2.createCLAHE(clipLimit=1.5,tileGridSize=(8,8)).apply(gray)
        return cv2.addWeighted(gray,.65,clahe,.35,0)
    raise ValueError(TILE_PREPROCESS)

# Correctly clipped binary mask -> single YOLO polygon per instance.
def tile_instance_mask(ann,x0,y0,side):
    seg=ann.get('segmentation',[])
    if isinstance(seg,dict):return full_mask(ann)[y0:y0+side,x0:x0+side].copy()
    mask=np.zeros((side,side),np.uint8)
    for p in paths(ann):
        q=np.rint(p-[x0,y0]).astype(np.int32)
        cv2.fillPoly(mask,[q],1)
    return mask

def mask_to_yolo_polygon(mask,side,min_area=85,epsilon=1.45):
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not contours:return None
    largest=max(contours,key=cv2.contourArea)
    if cv2.contourArea(largest)<min_area:return None
    approx=cv2.approxPolyDP(largest,epsilon,True).reshape(-1,2).astype(np.float64)
    if len(approx)<3:return None
    if len(approx)>300:approx=approx[np.linspace(0,len(approx)-1,300).astype(int)]
    approx[:,0]=np.clip(approx[:,0]/side,0,1);approx[:,1]=np.clip(approx[:,1]/side,0,1)
    if cv2.contourArea(approx.astype(np.float32))<1e-7:return None
    return '0 '+' '.join(f'{v:.6f}' for v in approx.ravel())

# Synthetic geometry contract: a filament spanning the tile edge must yield valid normalized coordinates.
unit=np.zeros((TILE_SIDE,TILE_SIDE),np.uint8)
cv2.rectangle(unit,(12,18),(220,62),1,-1)
assert mask_to_yolo_polygon(unit,TILE_SIDE) is not None
assert len(WINDOWS)==len(set(WINDOWS))
print('Tile geometry and instance-polygon unit tests passed',WINDOWS)

## 3 — Generate real native-tile training images and instance labels
We sample informative native windows for each training observation, including a center view and at least one diverse corner. Labels are fully clipped; small fragments are filtered. Outputs are deterministic and reusable from a previous V8 session. **Disk warning:** 513 observations × 4 JPEG tiles can occupy several GB.

In [ ]:
CONTRACT={'version':'v8-native-tile-train-v1','v4_sha256':V4_SHA,'split_sha256':SPLIT_SHA,
    'tile_side':TILE_SIDE,'tile_windows':TILE_WINDOWS,'train_views':TRAIN_VIEWS_PER_IMAGE,
    'val_views':VAL_VIEWS_PER_IMAGE,'tile_preprocess':TILE_PREPROCESS,
    'seed':SEED,'model_start':MODEL_START,'train_imgsz':TRAIN_IMGSZ}
CONTRACT_FILE=OUTPUT/'v8_contract.json'
if CONTRACT_FILE.exists():
    assert json.loads(CONTRACT_FILE.read_text())==CONTRACT,'Current V8 output belongs to a different experiment; change OUTPUT.'
else:CONTRACT_FILE.write_text(json.dumps(CONTRACT,indent=2))

if PREVIOUS_V8_ROOT is None:
    candidates=[p.parent for p in Path('/kaggle/input').rglob('v8_contract.json')] if Path('/kaggle/input').exists() else []
    candidates=[p for p in candidates if json.loads((p/'v8_contract.json').read_text())==CONTRACT]
    PREV=candidates[0] if len(candidates)==1 else None
else:
    PREV=Path(PREVIOUS_V8_ROOT)
    assert (PREV/'v8_contract.json').is_file(),PREV
    assert json.loads((PREV/'v8_contract.json').read_text())==CONTRACT,'Mismatched V8 checkpoint/data contract'
print('Previous compatible V8:',PREV)

TILE_DATA=OUTPUT/'tile_data'
YAML=TILE_DATA/'solar_tiles.yaml'

def choose_training_windows(stem,count):
    rec=REPS[stem]
    centers=[]
    for a in rec['anns']:
        pp=paths(a)
        if pp:
            q=np.concatenate(pp,0)
            centers.append(np.mean([q.min(axis=0),q.max(axis=0)],axis=0))
    scores=[sum(1 for c in centers if x<=c[0]<=x+TILE_SIDE and y<=c[1]<=y+TILE_SIDE) for x,y in WINDOWS]
    center=((W-TILE_SIDE)//2,(H-TILE_SIDE)//2)
    chosen=[center] if center in WINDOWS and count else []
    rest=[i for i in range(len(WINDOWS)) if WINDOWS[i] not in chosen]
    rng=random.Random(SEED+int(hashlib.sha256(stem.encode()).hexdigest()[:8],16))
    rng.shuffle(rest);rest.sort(key=lambda j:scores[j],reverse=True)
    for j in rest:
        if len(chosen)>=count:break
        chosen.append(WINDOWS[j])
    return chosen[:count]

def write_tile(gray,stem,win_id,x,y,rec,imdir,lbdir):
    name=f'{stem}__t{win_id}'
    dst=imdir/(name+'.jpg');txt=lbdir/(name+'.txt')
    if dst.is_file() and txt.is_file():return None
    if gray is None or gray.shape!=(H,W):raise ValueError(f'Bad native observation {stem}: {None if gray is None else gray.shape}')
    patch=tile_preprocess(gray[y:y+TILE_SIDE,x:x+TILE_SIDE])
    labels=[]
    for a in rec['anns']:
        mask=tile_instance_mask(a,x,y,TILE_SIDE)
        poly=mask_to_yolo_polygon(mask,TILE_SIDE)
        if poly:labels.append(poly)
    cv2.imwrite(str(dst),patch,[cv2.IMWRITE_JPEG_QUALITY,94])
    txt.write_text(('\n'.join(labels)+'\n') if labels else '')
    return len(labels)

def prepare_tiles():
    for subgroup,kind,num in [('train','train',TRAIN_VIEWS_PER_IMAGE),('val','tune',VAL_VIEWS_PER_IMAGE)]:
        imdir=TILE_DATA/'images'/subgroup;lbdir=TILE_DATA/'labels'/subgroup
        imdir.mkdir(parents=True,exist_ok=True);lbdir.mkdir(parents=True,exist_ok=True)
        for stem in tqdm(sorted(SPLIT[kind]),desc='Prepare '+subgroup):
            view_positions=choose_training_windows(stem,num)
            if all((imdir/f'{stem}__t{j}.jpg').is_file() and (lbdir/f'{stem}__t{j}.txt').is_file() for j in range(len(view_positions))):
                continue
            gray=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
            for j,(x,y) in enumerate(view_positions):
                write_tile(gray,stem,j,x,y,REPS[stem],imdir,lbdir)
    spec={'path':str(TILE_DATA),'train':'images/train','val':'images/val','nc':1,'names':{0:'filament'}}
    YAML.write_text(yaml.safe_dump(spec,sort_keys=False))
    counts={k:len(list((TILE_DATA/'images'/k).glob('*.jpg'))) for k in ('train','val')}
    print('Prepared tile samples',counts,'YAML',YAML)
    assert counts['train']>=len(SPLIT['train']) and counts['val']>=len(SPLIT['tune'])
    (OUTPUT/'preparation_report.json').write_text(json.dumps(counts,indent=2))
    return counts

if MODE in ('PREPARE','ALL','TRAIN'):
    if PREV and (PREV/'tile_data'/'solar_tiles.yaml').is_file() and not YAML.is_file():
        print('Previous tile_data attached; restoring image/label files for training...')
        for subset in ('train','val'):
            for folder in ('images','labels'):
                src=PREV/'tile_data'/folder/subset
                dest=TILE_DATA/folder/subset;dest.mkdir(parents=True,exist_ok=True)
                for f in src.glob('*'):
                    p=dest/f.name
                    if not p.exists():
                        try:os.symlink(f,p)
                        except OSError:shutil.copy2(f,p)
    prepare_tiles()
else:
    print('Training-data generation skipped in mode',MODE)

## 4 — Fine-tune a true tile-specialist detector on CPU
The previous V7 model was only *shown* native patches at inference. V8 actually backpropagates on those patches, starting from the existing trustworthy V4 detector. Gradient checkpoints are saved by Ultralytics. **Start with `MODE='TRAIN'`; on future sessions attach V8 outputs and rerun TRAIN to resume if incomplete.** This may require more than one CPU session.

In [ ]:
RUNS=OUTPUT/'runs';RUNS.mkdir(exist_ok=True)
RUN_DIR=RUNS/'tile_specialist'
BEST=RUN_DIR/'weights'/'best.pt'; LAST=RUN_DIR/'weights'/'last.pt'
TRAIN_DONE=RUN_DIR/'completed.json'

def restore_training_run():
    if not PREV:return
    src=PREV/'runs'/'tile_specialist'
    if src.is_dir() and not BEST.exists() and not LAST.exists():
        print('Restoring model training checkpoint from',src)
        shutil.copytree(src,RUN_DIR,dirs_exist_ok=True)
restore_training_run()

def train_tile_specialist():
    assert YAML.exists(),'Run MODE=PREPARE first, or MODE=TRAIN to prepare and train.'
    if TRAIN_DONE.exists() and BEST.exists():
        print('Reusing previously finished specialist:',BEST)
        return BEST
    resuming=LAST.exists()
    source=LAST if resuming else (V4_WEIGHTS if MODEL_START=='v4' else MODEL_START)
    print('RESUME from' if resuming else 'FINE-TUNE from',source)
    yolo=YOLO(str(source))
    common=dict(device='cpu',workers=TRAIN_WORKERS,amp=False,verbose=False,
                exist_ok=True,project=str(RUNS),name='tile_specialist',
                save=True,save_period=1,plots=False,cache=False)
    if resuming:
        # Ultralytics checkpoint contains original training arguments and optimizer state.
        # The train data must be attached at the same OUTPUT path (regenerated above).
        extra={'resume':True,'data':str(YAML),**common}
    else:
        extra=dict(data=str(YAML),epochs=TOTAL_EPOCHS,imgsz=TRAIN_IMGSZ,batch=TRAIN_BATCH,
              optimizer='AdamW',lr0=0.00012 if MODEL_START=='v4' else 0.00028,
              lrf=0.10,weight_decay=.0005,cos_lr=True,patience=TRAIN_PATIENCE,
              seed=SEED,deterministic=True,rect=False,overlap_mask=False,
              degrees=40,translate=.06,scale=.22,fliplr=.5,flipud=.5,
              mosaic=.12,close_mosaic=8,mixup=0.,copy_paste=0.,
              hsv_h=0.,hsv_s=0.,hsv_v=.08,box=7.5,cls=.5,dfl=1.5,
              **common)
    if TRAIN_TIME_HOURS>0:extra['time']=TRAIN_TIME_HOURS
    yolo.train(**extra)
    del yolo;gc.collect()
    if not BEST.is_file():raise FileNotFoundError('No best.pt written; inspect training log and saved last.pt')
    if not LAST.is_file():raise FileNotFoundError('Missing resumable last.pt checkpoint')
    # Preserve best weights even when the time cap ends before reaching TOTAL_EPOCHS.
    meta={'best_sha256':sha256(BEST),'last_sha256':sha256(LAST),
         'requested_epochs':TOTAL_EPOCHS,'per_session_cpu_hours_cap':TRAIN_TIME_HOURS,
         'base_model_start':MODEL_START,'data_contract':CONTRACT}
    (RUN_DIR/'last_training_session.json').write_text(json.dumps(meta,indent=2))
    # Only a full epoch-budget / early-stopped training run can be marked completed.
    try:
        pack=torch.load(LAST,map_location='cpu',weights_only=False)
        epoch=int(pack.get('epoch',-1))+1
        print('Last completed epoch:',epoch,'/',TOTAL_EPOCHS)
        if epoch>=TOTAL_EPOCHS or epoch>=TOTAL_EPOCHS-1:
            TRAIN_DONE.write_text(json.dumps({**meta,'epoch':epoch},indent=2))
    except Exception as e:print('Could not inspect completed epoch; restart-safe best.pt remains:',repr(e))
    return BEST

if MODE in ('TRAIN','ALL'):
    train_tile_specialist()
else:
    print('Training skipped. For actual performance improvement run PREPARE and then TRAIN.')

## 5 — Restore model for evaluation; cached live proposals tied to checkpoint hash
Cached proposals are allowed **only** with matching checkpoint SHA256 and inference geometry. No saved answer keys or precomputed external test submissions are read. V8 tile results are produced by the new model; original V4 full-disk results are preserved as the control.

In [ ]:
# The checkpoint may be inside a previously-attached V8 dataset in SCREEN/CONFIRM/SUBMIT.
if not BEST.is_file() and PREV is not None:
    candidate=PREV/'runs'/'tile_specialist'/'weights'/'best.pt'
    if candidate.exists():BEST=candidate

if MODE in ('SCREEN','CONFIRM','SUBMIT','ALL') and not BEST.exists():
    raise FileNotFoundError('No tile-specialist best.pt. First run PREPARE and TRAIN, save V8 outputs, attach them. '+str(BEST))
TILE_SHA=sha256(BEST) if BEST.exists() else None
print('New tile-specialist weights:',BEST if BEST.exists() else 'NOT TRAINED YET',TILE_SHA)

GLOBAL_DIR=OUTPUT/'global';TILE_DIR=OUTPUT/'tile_proposals'
GLOBAL_DIR.mkdir(exist_ok=True);TILE_DIR.mkdir(exist_ok=True)
V4_NET=None;TILE_NET=None

def net_for(kind):
    global V4_NET,TILE_NET
    if kind=='global':
        if V4_NET is None:V4_NET=YOLO(str(V4_WEIGHTS))
        return V4_NET
    if TILE_NET is None:
        if not BEST.exists():raise FileNotFoundError('Tile detector not trained')
        TILE_NET=YOLO(str(BEST))
    return TILE_NET

def meta_for(kind,stem):
    return {'source':'LIVE_H_ALPHA_IMAGE_INFERENCE','stem':stem,
            'model_sha256':V4_SHA if kind=='global' else TILE_SHA,
            'inference_imgsz':GLOBAL_IMGSZ if kind=='global' else PREDICT_IMAGE_SIZE,
            'raw_conf':RAW_CONF,'raw_iou':RAW_IOU,'max_det':MAX_DET,
            'kind':kind,'tile_side':TILE_SIDE if kind=='tile' else None,
            'preprocess':TILE_PREPROCESS if kind=='tile' else 'none',
            'windows':WINDOWS if kind=='tile' else None}

def load_cached(kind,stem):
    folder=GLOBAL_DIR if kind=='global' else TILE_DIR
    candidates=[folder/(stem+'.json')]
    if PREV:candidates.append(PREV/('global' if kind=='global' else 'tile_proposals')/(stem+'.json'))
    for file in candidates:
        if not file.is_file():continue
        try:
            value=json.loads(file.read_text())
            if all(value.get(k)==v for k,v in meta_for(kind,stem).items()):return value['proposals']
        except Exception:pass
    if kind=='global':
        old=V4/'cache'/'yolo11n-seg'/(stem+'.json')
        if old.is_file():
            try:
                val=json.loads(old.read_text())
                if (val.get('stem')==stem and val.get('checkpoint_sha256')==V4_SHA
                    and val.get('predict_imgsz')==GLOBAL_IMGSZ and val.get('tta') is False
                    and val.get('source')=='LIVE_IMAGE_INFERENCE'):
                    return val['proposals']
            except Exception:pass
    return None

@torch.inference_mode()
def infer_one_view(source,kind,x=0,y=0):
    # source grayscale; tile specialist sees the exact training-time image transform
    if kind=='tile':source=tile_preprocess(source)
    img=cv2.cvtColor(source,cv2.COLOR_GRAY2BGR)
    output=net_for(kind).predict(source=img,
        imgsz=GLOBAL_IMGSZ if kind=='global' else PREDICT_IMAGE_SIZE,
        conf=RAW_CONF,iou=RAW_IOU,retina_masks=True,agnostic_nms=True,
        max_det=MAX_DET,device='cpu',verbose=False)[0]
    if output.masks is None or output.boxes is None:return []
    scores=output.boxes.conf.detach().cpu().numpy()
    masks=output.masks.data.detach().cpu().numpy()
    assert len(scores)==len(masks)
    proposals=[]
    for confidence,m in zip(scores,masks):
        local=(m>.5).astype(np.uint8)
        if local.shape!=source.shape:local=cv2.resize(local,(source.shape[1],source.shape[0]),interpolation=cv2.INTER_NEAREST)
        if not local.any():continue
        if kind=='tile':
            touches=(bool(local[:12].any() and y>0) or
                     bool(local[-12:].any() and y+TILE_SIDE<H) or
                     bool(local[:,:12].any() and x>0) or
                     bool(local[:,-12:].any() and x+TILE_SIDE<W))
            full=np.zeros((H,W),np.uint8);full[y:y+TILE_SIDE,x:x+TILE_SIDE]=local
        else:
            full=local;touches=False
        rr=encoded(full)
        proposals.append({'score':float(confidence),'area':int(coco.area(rr)),
                          'counts':asc(rr),'origin':kind,'edge':touches,'x':x,'y':y})
    return proposals

def ensure_preds(kind,stems,lookup):
    missing=[s for s in stems if load_cached(kind,s) is None]
    print(kind,'found',len(stems)-len(missing),'missing',len(missing))
    if not missing:return
    folder=GLOBAL_DIR if kind=='global' else TILE_DIR
    for stem in tqdm(missing,desc='CPU '+kind+' inference'):
        im=cv2.imread(str(lookup[stem]),cv2.IMREAD_GRAYSCALE)
        if im is None or im.shape!=(H,W):raise RuntimeError('Invalid official image: '+str(lookup[stem]))
        proposals=[]
        if kind=='global':proposals=infer_one_view(im,'global')
        else:
            for x,y in WINDOWS:
                tile=im[y:y+TILE_SIDE,x:x+TILE_SIDE]
                proposals.extend(infer_one_view(tile,'tile',x,y))
        payload={**meta_for(kind,stem),'proposals':proposals}
        (folder/(stem+'.json')).write_text(json.dumps(payload,separators=(',',':')))

def get_pool(stem,mode):
    globalp=load_cached('global',stem)
    if globalp is None:raise FileNotFoundError('Missing original YOLO global proposals: '+stem)
    gg=[{**p,'origin':'global','edge':False} for p in globalp]
    if mode=='global':return gg
    tp=load_cached('tile',stem)
    if tp is None:raise FileNotFoundError('Missing new tile-specialist inference: '+stem)
    tt=[{**p,'origin':'tile','edge':bool(p.get('edge',False))} for p in tp]
    return tt if mode=='tile' else gg+tt

## 6 — Stronger instance fusion and exact V4 control
All area/confidence filtering occurs *before* greedy pixel allocation. Tile-edge fragments can be rejected, overlapping instances are deduplicated using both IoU and containment, and the final masks are disjoint. The original V4 pipeline is reproduced exactly as a fixed control.

In [ ]:
def disjoint(rles):
    if len(rles)<2:return True
    v=np.asarray(coco.iou(rles,rles,[0]*len(rles)),dtype=np.float64)
    np.fill_diagonal(v,0.)
    return bool(np.all(v<=0))

def v4_control(stem):
    raw=load_cached('global',stem)
    assert raw is not None
    ranked=sorted([p for p in raw if float(p['score'])>=.32 and int(p['area'])>=350],
                  key=lambda p:(-float(p['score']),-int(p['area'])))
    occupied=np.zeros((H,W),np.uint8);prior=[];out=[]
    for p in ranked:
        rr=as_rle(p['counts'])
        if prior:
            sim=np.asarray(coco.iou([rr],prior,[0]*len(prior)))[0]
            if sim.size and sim.max()>=.50:continue
        prior.append(rr)
        full=coco.decode(rr).astype(bool)&(occupied==0)
        if full.sum()<350:continue
        occupied[full]=1;out.append(encoded(full))
    return out

def apply_recipe(stem,cfg):
    if cfg=='v4':return v4_control(stem)
    mode=cfg['mode']
    vals=get_pool(stem,mode)
    kept=[]
    for p in vals:
        is_tile=p['origin']=='tile'
        if p['score'] < (cfg['tile_conf'] if is_tile else cfg['global_conf']):continue
        if p['area']<cfg['min_area']:continue
        if is_tile and cfg['seam_guard'] and p['edge']:continue
        rank=p['score']+(cfg['tile_bonus'] if is_tile else 0.)
        kept.append((rank,p))
    kept.sort(key=lambda t:(-t[0],-t[1]['area']))
    occupied=np.zeros((H,W),np.uint8);used=[];output=[]
    for _,p in kept:
        rr=as_rle(p['counts'])
        if used:
            scores=np.asarray(coco.iou([rr],used,[0]*len(used)),dtype=np.float64)[0]
            usedareas=np.asarray(coco.area(used),dtype=np.float64)
            intersect=scores*(usedareas+p['area'])/(1+scores+1e-9)
            containment=intersect/max(1,p['area'])
            if np.any(scores>=cfg['nms_iou']) or np.any(containment>=cfg['containment']):continue
        binary=coco.decode(rr).astype(bool)
        free=binary&(occupied==0)
        pix=int(free.sum())
        if pix<cfg['min_area'] or pix/max(1,p['area'])<cfg['min_free_fraction']:continue
        occupied[free]=1;used.append(rr);output.append(encoded(free))
    assert disjoint(output)
    return output

# Toy overlapping instances: containment must not produce duplicate masks.
toy1=np.zeros((H,W),np.uint8);toy1[30:120,20:200]=1
toy2=np.zeros((H,W),np.uint8);toy2[45:85,55:120]=1
assert not disjoint([encoded(toy1),encoded(toy2)])
assert disjoint([encoded(toy1)])
del toy1,toy2
print('Geometry/NMS tests passed.')

## 7 — Multi-annotator, one-to-one instance PQ and error counts
Use all organizer-permitted annotation versions for each validation observation; IoU > 0.5 matching uses one-to-one assignment. The older notebook's sum of all IoUs exceeding .5 can double-count overlapping GT, so the new strict metric is shown separately. The official evaluator remains authoritative.

In [ ]:
TRUTH={}
def gt_masks(stem):
    if stem not in TRUTH:
        views=[]
        for rec in OBS[stem]:
            rles=[]
            for ann in rec['anns']:
                seg=ann.get('segmentation',[])
                if isinstance(seg,dict):
                    r=as_rle(seg) if not isinstance(seg.get('counts'),list) else coco.frPyObjects(seg,H,W)
                else:
                    polys=[np.clip(p,[0,0],[W-1,H-1]).astype(np.float64).ravel().tolist() for p in paths(ann)]
                    if not polys:continue
                    shapes=coco.frPyObjects(polys,H,W)
                    r=coco.merge(shapes) if isinstance(shapes,list) else shapes
                rles.append(r)
            views.append((rec['image_id'],rles))
        TRUTH[stem]=views
    return TRUTH[stem]

def match_instances(gt,pred):
    if not gt and not pred:return np.zeros(6,dtype=np.float64)
    if not gt:return np.array([0,0,len(pred),0,0,0],dtype=np.float64)
    if not pred:return np.array([0,0,0,len(gt),0,0],dtype=np.float64)
    iou=np.asarray(coco.iou(gt,pred,[0]*len(pred)),dtype=np.float64)
    rr,cc=linear_sum_assignment(-iou)
    valid=iou[rr,cc]>.5
    tp=int(valid.sum());fp=len(pred)-tp;fn=len(gt)-tp
    one2many=int(((iou>.1).sum(axis=1)>1).sum())
    many2one=int(((iou>.1).sum(axis=0)>1).sum())
    return np.array([float(iou[rr[valid],cc[valid]].sum()),tp,fp,fn,one2many,many2one],dtype=np.float64)

def pq_stats(counts):
    summ,tp,fp,fn,one,many=map(float,counts)
    denom=tp+.5*(fp+fn)
    return {'pq':float(summ/denom) if denom else 0.,
            'sq':float(summ/tp) if tp else 0.,
            'rq':float(tp/denom) if denom else 0.,
            'tp':int(tp),'fp':int(fp),'fn':int(fn),
            'one_to_many':int(one),'many_to_one':int(many)}

def score_recipe(cfg,stems,save_details=False):
    summed=np.zeros(6,dtype=np.float64);rows=[]
    for stem in stems:
        pred=apply_recipe(stem,cfg)
        for version,gt in gt_masks(stem):
            v=match_instances(gt,pred);summed+=v
            if save_details:rows.append({'stem':stem,'version':version,**pq_stats(v)})
    return pq_stats(summed),pd.DataFrame(rows)

m=np.zeros((H,W),np.uint8);m[32:120,40:170]=1;rr=encoded(m)
assert abs(pq_stats(match_instances([rr],[rr]))['pq']-1)<1e-8
assert pq_stats(match_instances([rr],[]))['fn']==1
assert pq_stats(match_instances([],[rr]))['fp']==1
print('One-to-one PQ sanity checks passed; this is not an official leaderboard score.')

## 8 — Optional exact V7 baseline (the user's reported 0.34 public recipe)
If the old V7 run was saved, replay its **actual selected parameters and its own real cached tile detections**. If the V7 output was not retained, it cannot be faithfully reconstructed without running the old checkpoint on every native tile again. In that case V4 remains the formal internal control; the user-reported 0.34 remains the external leaderboard reference.

In [ ]:
V7ROOT=Path(SOURCE_V7_ROOT) if SOURCE_V7_ROOT is not None else None
if not V7ROOT or not (V7ROOT/'v7_config.json').is_file():
    candidates=[p.parent for p in Path('/kaggle/input').rglob('v7_config.json')] if Path('/kaggle/input').exists() else []
    matching=[]
    for p in candidates:
        try:
            c=json.loads((p/'v7_config.json').read_text())
            if c.get('checkpoint_sha256')==V4_SHA and c.get('split_sha256')==SPLIT_SHA:matching.append(p)
        except Exception:pass
    V7ROOT=matching[0] if len(matching)==1 else None
V7CHOICE=None;V7CONFIG=None
if V7ROOT and (V7ROOT/'v7_config.json').exists():
    V7CONFIG=json.loads((V7ROOT/'v7_config.json').read_text())
    if V7CONFIG.get('checkpoint_sha256')!=V4_SHA or V7CONFIG.get('split_sha256')!=SPLIT_SHA:
        print('V7 provenance does not match this trained checkpoint. Ignoring V7.')
        V7ROOT=None
    elif (V7ROOT/'selection.json').is_file():
        V7CHOICE=json.loads((V7ROOT/'selection.json').read_text()).get('strategy')
print('V7 replay:',V7ROOT,'selected recipe:',V7CHOICE)

def old_v7_tiles(stem):
    if not V7ROOT:return None
    file=V7ROOT/'tiles'/(stem+'.json')
    if not file.is_file():return None
    try:
        data=json.loads(file.read_text())
        for k,v in V7CONFIG.items():
            if data.get(k)!=v:return None
        if data.get('stem')!=stem or data.get('kind')!='tile' or data.get('source')!='LIVE_TEST_OR_TRAIN_JPEG_INFERENCE':return None
        return [{**p,'origin':'tile','edge':bool(p.get('truncated',False))} for p in data['proposals']]
    except Exception:return None

def replay_v7(stem):
    if V7CHOICE=='v4':return v4_control(stem)
    if not isinstance(V7CHOICE,dict):raise RuntimeError('V7 selected recipe unavailable')
    pp=V7CHOICE
    glob=[{**p,'origin':'global','edge':False} for p in (load_cached('global',stem) or [])]
    tiles=old_v7_tiles(stem)
    if pp['mode']!='global' and tiles is None:raise FileNotFoundError('V7 source tiles missing for '+stem)
    pool=glob if pp['mode']=='global' else glob+tiles
    candidates=[]
    for q in pool:
        istile=q['origin']=='tile'
        if q['score'] < (pp['tile_conf'] if istile else pp['global_conf']):continue
        if q['area']<pp['min_area']:continue
        if istile and pp['seam_guard'] and q['edge']:continue
        candidates.append({**q,'rank':float(q['score'])+(pp['tile_bonus'] if istile else 0.)})
    candidates.sort(key=lambda q:(-q['rank'],-q['area']))
    occupied=np.zeros((H,W),np.uint8);previous=[];outputs=[]
    for q in candidates:
        rr=as_rle(q['counts'])
        if previous:
            ov=np.asarray(coco.iou([rr],previous,[0]*len(previous)))[0]
            ar=np.asarray(coco.area(previous),dtype=np.float64)
            inter=ov*(ar+q['area'])/(1+ov+1e-9)
            if np.any(ov>=pp['nms_iou']) or np.any(inter/max(1,q['area'])>=pp['containment']):continue
        binary=coco.decode(rr).astype(bool)
        free=binary&(occupied==0)
        if free.sum()<pp['min_area'] or free.sum()/max(1,q['area'])<pp['min_free_fraction']:continue
        occupied[free]=1;previous.append(rr);outputs.append(encoded(free))
    assert disjoint(outputs)
    return outputs

# Safely expose V7 as one additional contender without modifying V8 inference code.
_apply_v8_only=apply_recipe
def apply_recipe(stem,cfg):
    if cfg=='v7':return replay_v7(stem)
    return _apply_v8_only(stem,cfg)

def v7_complete(stems):
    if not V7ROOT or V7CHOICE is None:return False
    if V7CHOICE=='v4':return True
    if isinstance(V7CHOICE,dict) and V7CHOICE.get('mode')=='global':return True
    return all(old_v7_tiles(stem) is not None for stem in stems)

## 9 — Screen tile-specialist candidates on 28 tuning images
The new model gets three fair chances: **tile alone**, **global + tile**, or the old full-disk control. The original 0.34 V7 run is added if its entire cached sample exists. CPU evaluation caches model predictions, not labels for any test image.

In [ ]:
SCREEN_FILE=OUTPUT/'screening_results.csv'
SCREEN_META=OUTPUT/'screening_meta.json'
FULL_FILE=OUTPUT/'full_tune_results.csv'
CHOSEN_FILE=OUTPUT/'selected_pipeline.json'
AUDIT_FILE=OUTPUT/'audit_comparison.json'
rng=random.Random(SEED+47)
SAMPLES=sorted(SPLIT['tune']);rng.shuffle(SAMPLES)
SCREEN_STEMS=sorted(SAMPLES[:min(SCREEN_COUNT,len(SAMPLES))])
TUNE_STEMS=sorted(SPLIT['tune'])

def candidate_grid():
    # Stricter thresholds than the original .1 inference gate; search focuses on recognition, not noise.
    yield from (
        {'mode':mode,'global_conf':gc,'tile_conf':tc,'min_area':area,'seam_guard':guard,
         'tile_bonus':bonus,'nms_iou':.55,'containment':.88,'min_free_fraction':.50}
        for mode,gc,tc,area,guard,bonus in product(
            ('tile','hybrid'),(.30,.42),(.30,.45,.60),(250,420),(False,True),(0.,.08))
        if not (mode=='tile' and bonus!=0.)
    )
    # Global no-tile candidate detects changes from selection policy rather than new training.
    yield {'mode':'global','global_conf':.32,'tile_conf':1.0,'min_area':350,
           'seam_guard':False,'tile_bonus':0.,'nms_iou':.50,
           'containment':.88,'min_free_fraction':.50}
CANDIDATES=list(candidate_grid())
print('Candidate recipes:',len(CANDIDATES))

def run_screen():
    ensure_preds('global',SCREEN_STEMS,TRAIN)
    ensure_preds('tile',SCREEN_STEMS,TRAIN)
    board=[]
    base,_=score_recipe('v4',SCREEN_STEMS)
    board.append({'index':-1,'method':'v4',**base})
    if v7_complete(SCREEN_STEMS):
        s,_=score_recipe('v7',SCREEN_STEMS)
        board.append({'index':-2,'method':'v7',**s})
    for i,p in enumerate(tqdm(CANDIDATES,desc='True PQ native-tile screen')):
        s,_=score_recipe(p,SCREEN_STEMS)
        board.append({'index':i,'method':p['mode'],**p,**s})
    df=pd.DataFrame(board).sort_values(['pq','rq'],ascending=False)
    df.to_csv(SCREEN_FILE,index=False)
    SCREEN_META.write_text(json.dumps({'tile_model_sha256':TILE_SHA,'v4_sha256':V4_SHA,'split_sha256':SPLIT_SHA,'contract':CONTRACT},indent=2))
    print('Stage 1 control PQ:',base['pq'])
    display(df.head(16))
    return df

if MODE in ('SCREEN','ALL'):
    screen=run_screen()
elif MODE in ('CONFIRM','SUBMIT'):
    source=SCREEN_FILE if SCREEN_FILE.exists() else (PREV/'screening_results.csv' if PREV else None)
    meta_source=SCREEN_META if SCREEN_META.exists() else (PREV/'screening_meta.json' if PREV else None)
    expected_meta={'tile_model_sha256':TILE_SHA,'v4_sha256':V4_SHA,'split_sha256':SPLIT_SHA,'contract':CONTRACT}
    valid_screen=bool(source and source.is_file() and meta_source and meta_source.is_file() and json.loads(meta_source.read_text())==expected_meta)
    if valid_screen:screen=pd.read_csv(source)
    elif MODE=='CONFIRM':
        print('No screening results found; running SCREEN using this checkpoint before CONFIRM')
        screen=run_screen()
    else:raise FileNotFoundError('Run SCREEN and CONFIRM before SUBMIT')
else:
    print('Screening not requested in this mode')

## 10 — Confirm only finalists on all 97 tune images, then report historical audit
The historical audit has been inspected in earlier experiments. We include it only as a **regression diagnostic**, not as fresh independent evidence. The selected strategy is stored with both original and new checkpoint hashes. New inference must beat the available legacy control by a meaningful tune PQ margin to be promoted.

In [ ]:
def chosen_cfg_from_index(i):
    if i==-1:return 'v4'
    if i==-2:return 'v7'
    return CANDIDATES[i]

def run_confirm():
    ensure_preds('global',TUNE_STEMS,TRAIN)
    ensure_preds('tile',TUNE_STEMS,TRAIN)
    indices=[int(i) for i in screen['index'] if int(i)>=0][:SCREEN_TOP]
    # Always include tile-only and global-only controls in confirm.
    for mode in ('tile','global'):
        match=next((i for i,p in enumerate(CANDIDATES) if p['mode']==mode),None)
        if match is not None:indices.append(match)
    indices=list(dict.fromkeys([-1]+([-2] if v7_complete(TUNE_STEMS) else [])+indices))
    rows=[]
    for i in tqdm(indices,desc='Final 97-image tune PQ'):
        cfg=chosen_cfg_from_index(i)
        values,_=score_recipe(cfg,TUNE_STEMS)
        rows.append({'index':i,'method':cfg if isinstance(cfg,str) else cfg['mode'],**values})
    table=pd.DataFrame(rows).sort_values(['pq','rq'],ascending=False)
    table.to_csv(FULL_FILE,index=False)
    ref=table[table['index'].isin([-1,-2])].sort_values('pq',ascending=False).iloc[0]
    legacy=float(ref['pq']);ref_cfg=chosen_cfg_from_index(int(ref['index']))
    top=table.iloc[0];win_cfg=chosen_cfg_from_index(int(top['index']))
    improvement=float(top['pq']-legacy)
    selected=win_cfg if (isinstance(win_cfg,dict) and improvement>=PROMOTION_MARGIN) else ref_cfg
    chosen={'selected':selected,'full_tune_best_pq':float(top['pq']),
            'legacy_control_pq':legacy,'gain_over_best_legacy':improvement,
            'minimum_gain_to_promote':PROMOTION_MARGIN,
            'model_sha256':TILE_SHA,'v4_model_sha256':V4_SHA,'split_sha256':SPLIT_SHA,
            'tile_config':CONTRACT,'screen_images':SCREEN_STEMS,
            'note':'Selected only on original training-side tune set; historical audit not pristine'}
    CHOSEN_FILE.write_text(json.dumps(chosen,indent=2))
    print('FULL TUNE RESULTS');display(table)
    print('Selected strategy:',selected,'gap vs legacy:',improvement)
    if RUN_OLD_AUDIT:
        audits=sorted(SPLIT['audit'])
        ensure_preds('global',audits,TRAIN)
        # No needless specialist inference if selected the original V4/V7 baseline.
        if isinstance(selected,dict):ensure_preds('tile',audits,TRAIN)
        models={'v4': 'v4','selected':selected}
        if v7_complete(audits):models['v7']='v7'
        audit={'status':'PREVIOUSLY_SEEN_AUDIT_NOT_NEW_INDEPENDENT_EVIDENCE','scores':{}}
        for name,cfg in models.items():
            values,details=score_recipe(cfg,audits,save_details=(name=='selected'))
            audit['scores'][name]=values
            if name=='selected':details.to_csv(OUTPUT/'audit_by_annotator.csv',index=False)
        AUDIT_FILE.write_text(json.dumps(audit,indent=2))
        print('Historical audit comparison:');print(json.dumps(audit,indent=2))
    return chosen

if MODE in ('CONFIRM','ALL'):
    CHOICE=run_confirm()
elif MODE=='SUBMIT':
    src=CHOSEN_FILE if CHOSEN_FILE.exists() else (PREV/'selected_pipeline.json' if PREV else None)
    if src is None or not src.is_file():raise FileNotFoundError('CONFIRM output selected_pipeline.json missing')
    CHOICE=json.loads(src.read_text())
else:CHOICE=None
if CHOICE:
    assert CHOICE['model_sha256']==TILE_SHA and CHOICE['v4_model_sha256']==V4_SHA
    assert CHOICE['split_sha256']==SPLIT_SHA and CHOICE['tile_config']==CONTRACT

## 11 — Live 180-image inference, strict COCO RLE and a single selected CSV
For a promoted V8 recipe, this does CPU inference on five full-resolution windows per test image using your **new fine-tuned model**, plus the original full disk where relevant. Cache images as you proceed; recover them across sessions with `PREVIOUS_V8_ROOT`. A legitimate untouched Kaggle score will not be known until you submit the generated CSV.

In [ ]:
SUB_PATH=Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else OUTPUT/'submission.csv'

def make_submission():
    if not CHOICE:raise RuntimeError('Run SCREEN, then CONFIRM and save selections before SUBMIT')
    cfg=CHOICE['selected']
    stems=sorted(TEST)
    ensure_preds('global',stems,TEST)
    if isinstance(cfg,dict):ensure_preds('tile',stems,TEST)
    if cfg=='v7' and not v7_complete(stems):
        raise FileNotFoundError('Selected original V7 control but its test-tile caches are missing. Attach V7 submission output.')
    rows=[];stats=[]
    for stem in tqdm(stems,desc='Create competition-compliant instance masks'):
        rles=apply_recipe(stem,cfg)
        assert disjoint(rles),f'Overlapping final instances for {stem}'
        for j,rr in enumerate(rles,1):
            if coco.area(rr)<=0:raise RuntimeError('Zero-area RLE must never be submitted')
            rows.append((f'{stem}_{j}',asc(rr)))
        stats.append({'stem':stem,'instances':len(rles)})
    out=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    assert len(out)>0 and out['filament_id'].is_unique and not out.isna().any().any()
    out.to_csv(SUB_PATH,index=False)
    check=pd.read_csv(SUB_PATH,keep_default_na=False)
    assert list(check.columns)==['filament_id','segmentation_rle']
    for stem,g in check.groupby(check.filament_id.str.rsplit('_',n=1).str[0]):
        assert stem in TEST
        each=[as_rle(x) for x in g['segmentation_rle']]
        assert disjoint(each) and all(coco.area(x)>0 for x in each)
    stats=pd.DataFrame(stats)
    stats.to_csv(OUTPUT/'test_instance_counts.csv',index=False)
    meta={'output':str(SUB_PATH),'submission_sha256':sha256(SUB_PATH),'rows':len(out),
          'test_observations':len(TEST),'observations_with_predictions':int((stats.instances>0).sum()),
          'strategy':cfg,'v4_checkpoint_sha256':V4_SHA,'v8_checkpoint_sha256':TILE_SHA,
          'producer':'LIVE_CPU_INFERENCE_ON_OFFICIAL_H_ALPHA_IMAGES',
          'test_labels_used':False,'precompiled_submission_payload_used':False}
    (OUTPUT/'submission_report.json').write_text(json.dumps(meta,indent=2))
    print('SUBMISSION READY',json.dumps(meta,indent=2))
    return meta

if MODE in ('SUBMIT','ALL'):
    make_submission()
else:print('No test submission in this mode; switch to SUBMIT after confirmed validation.')

## 12 — Scientific diagnosis and recovery instructions

1. **First session:** `MODE='PREPARE'` creates native training tiles. Save `/kaggle/working/solar_cpu_v8` as a Kaggle dataset after completion. Some V8 filenames are large; check Kaggle disk quota.
2. **Training session:** Attach that V8 dataset, point `PREVIOUS_V8_ROOT` to its actual folder if it isn't autodetected, `MODE='TRAIN'`. CPU training is materially slower than the old 4-minute embedded-payload notebook. Save V8 outputs again after a training segment. For later training, reattach the most recent V8 checkpoint, not a stale version.
3. **Selection session:** `MODE='SCREEN'`, save outputs; then `MODE='CONFIRM'` with that output attached. We compare real PQ on all 97 original tune images and, optionally, replay a saved V7 run as the **0.34 external-control candidate**.
4. **Submission session:** `MODE='SUBMIT'` with both models' outputs attached. This generates `/kaggle/working/submission.csv`. Submit and report `selected_pipeline.json`, `full_tune_results.csv`, `audit_comparison.json`, and official public PQ.

**Why this is a real change:** Original V4 YOLO11n was trained mostly on full disks (substantially downscaled); original V7 simply tiled *its inference*. V8 changes the input distribution during **supervised training** using native 1280-pixel crops. It can learn finer structures, but stronger recognition is not guaranteed. The competition also considers qualitative morphology, over/under-segmentation and reproducibility.

**Do not infer success from training loss or the number of predicted rows.** Reproduction of the 0.55 embedded payload is not a supported scientific baseline. The organizer's private/qualitative evaluation, not public-score chasing, determines a legitimate winner.